# Rice Disease Detection — AI AgriData Intelligence Race (TELEPATI 8.0)

Notebook ini memuat seluruh alur pengembangan model deteksi penyakit padi, mulai dari dataset mentah resmi sampai inference dan perhitungan skor.

| | |
|---|---|
| Tugas | Object detection 11 kelas (10 penyakit + Healthy), citra 640×640 |
| Data | Dataset resmi Agritech Data (format COCO, split train/valid/test) |
| Model | Ultralytics YOLO, dibangun dari file `.yaml` dan dilatih dari nol |
| Metrik | F1-Score makro pada split test resmi |
| Platform | Google Colab, GPU T4 |

**Mode eksekusi**

| `MODE` | Yang dijalankan | Kegunaan |
|---|---|---|
| `"full"` | Bagian 1–10 | Reproduksi lengkap: data mentah → training → evaluasi |
| `"inference_only"` | Bagian 1–3, lalu 11 | Reproduksi skor dari weights rilis, tanpa training |

**Kepatuhan aturan**

- Seluruh pemrosesan data dimulai dari zip dataset mentah dan dikerjakan di notebook ini.
- Training hanya memakai split **train**. Split **valid** dipakai untuk memilih checkpoint. Split **test** hanya dipakai di bagian 10 (sekali) dan bagian 11.
- Arsitektur dibangun dari file `.yaml` dengan `pretrained=False`. Tidak ada bobot eksternal yang dipakai untuk training.
- Seed tetap (`SEED = 42`) dan mode deterministik di setiap tahap.

## 0. Konfigurasi

Semua parameter yang perlu diubah ada di sel ini. Sel lain tidak perlu disunting.

- `RUN_ID` memilih konfigurasi training dari tabel `RUNS` di bagian 7.
- `WEIGHTS_URL`, `WEIGHTS_SHA256`, dan `E2_CONF` diisi setelah model final dirilis.
- `RUN_FINAL_TEST` dibiarkan `False` sampai model final dipilih dari hasil valid.

In [ ]:
MODE = "full"  # "full" | "inference_only"
RUN_ID = "A_yolo11s_s42"

DATA_ZIP = "/content/drive/MyDrive/telepati/data/telepati-agridata-raw.zip"
DATA_ZIP_SHA256 = "797e47749f91f4403e778dd0150716da1cd94cddbecee2ec56cf5545c50cec50"

RUNS_DIR = "/content/drive/MyDrive/telepati/runs"
WORK_DIR = "/content/work"

WEIGHTS_URL = ""
WEIGHTS_SHA256 = ""
E2_CONF = None

SEED = 42
IMGSZ = 640
EVAL_BATCH = 16
RUN_FINAL_TEST = False

## 1. Lingkungan

Memasang versi library yang dipin, menghubungkan Google Drive (khusus Colab), lalu mengunci seed.

Catatan: saat training dengan AMP, Ultralytics mengunduh model kecil hanya untuk *memeriksa* dukungan mixed precision pada GPU. Bobot tersebut tidak dipakai sebagai inisialisasi; model tetap dibangun dari `.yaml`.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.184"], check=True)
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import hashlib
import json
import math
import os
import random
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

WORK_DIR = Path(WORK_DIR)
RUNS_DIR = Path(RUNS_DIR)
RAW_DIR = WORK_DIR / "raw"
YOLO_DIR = WORK_DIR / "yolo"
SPLITS = ("train", "valid", "test")

print(f"python      {sys.version.split()[0]}")
print(f"torch       {torch.__version__}")
print(f"ultralytics {ultralytics.__version__}")
print(f"device      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## 2. Dataset mentah

Zip berisi folder `train/`, `valid/`, dan `test/` persis seperti dataset resmi (gambar `.jpg` + `_annotations.coco.json`). Checksum diverifikasi sebelum ekstraksi supaya data yang dipakai identik dengan data resmi.

In [ ]:
EXPECTED_COUNTS = {
    "train": {"images": 10133, "annotations": 20163},
    "valid": {"images": 2106, "annotations": 4888},
    "test": {"images": 1059, "annotations": 2670},
}


def sha256sum(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_dataset(zip_path: Path, dest: Path, expected_sha256: str) -> None:
    actual = sha256sum(zip_path)
    if actual != expected_sha256:
        raise ValueError(f"Checksum dataset tidak cocok: {actual}")
    if all((dest / split / "_annotations.coco.json").exists() for split in SPLITS):
        print(f"Dataset sudah terekstrak di {dest}")
        return
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    print(f"Dataset diekstrak ke {dest}")


def load_coco(split: str) -> dict:
    with open(RAW_DIR / split / "_annotations.coco.json", encoding="utf-8") as fh:
        return json.load(fh)


extract_dataset(Path(DATA_ZIP), RAW_DIR, DATA_ZIP_SHA256)
coco = {split: load_coco(split) for split in SPLITS}

for split, expected in EXPECTED_COUNTS.items():
    n_images, n_annotations = len(coco[split]["images"]), len(coco[split]["annotations"])
    assert n_images == expected["images"], f"{split}: {n_images} gambar"
    assert n_annotations == expected["annotations"], f"{split}: {n_annotations} anotasi"
    print(f"{split:<6} {n_images:>6} gambar  {n_annotations:>6} anotasi")

## 3. Pemetaan 21 kategori → 11 kelas canonical

File COCO bawaan memiliki 21 kategori: 18 nama kelas dengan variasi penulisan dan 3 supercategory (`Leaf-blight`, `Rice-Leaf-Diseasee`, `paddy`) yang tidak memiliki bounding box. Pemetaan dilakukan berdasarkan nama kategori, lalu divalidasi agar tidak ada anotasi yang tertinggal.

Urutan kelas mengikuti urutan alfabet pada halaman regulasi dan dipakai konsisten di training, evaluasi, dan inference.

In [ ]:
CLASS_NAMES = [
    "Bacterial leaf blight",
    "Bacterial panicle blight",
    "Blast",
    "Brown spot",
    "False smut",
    "Healthy",
    "Leaf roller",
    "Leaf scald",
    "Narrow brown",
    "Sheath blight",
    "Tungro",
]

RAW_TO_CANONICAL = {
    "Bacterial leaf blight": "Bacterial leaf blight",
    "Bacterial panicle Blight": "Bacterial panicle blight",
    "Blast": "Blast",
    "Leaf blast": "Blast",
    "Infected Blast": "Blast",
    "BrownSpot": "Brown spot",
    "Brown spot": "Brown spot",
    "False-Smut": "False smut",
    "Healthy Rice Leaf": "Healthy",
    "Healthy Rice beads": "Healthy",
    "Healthy": "Healthy",
    "healthy": "Healthy",
    "Leaf-roller": "Leaf roller",
    "Leaf Scald": "Leaf scald",
    "Leaf scald": "Leaf scald",
    "Narrow brown": "Narrow brown",
    "Sheath Blight": "Sheath blight",
    "Rice-Tungro": "Tungro",
}
SUPERCATEGORIES = {"Leaf-blight", "Rice-Leaf-Diseasee", "paddy"}
CLASS_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def to_frames(data: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Returns (images, boxes); boxes use canonical class ids and COCO xywh in pixels.
    category_names = {c["id"]: c["name"] for c in data["categories"]}
    unknown = set(category_names.values()) - set(RAW_TO_CANONICAL) - SUPERCATEGORIES
    if unknown:
        raise ValueError(f"Kategori tanpa pemetaan: {sorted(unknown)}")

    images = pd.DataFrame(data["images"])[["id", "file_name", "width", "height"]]
    rows = []
    for ann in data["annotations"]:
        raw_name = category_names[ann["category_id"]]
        if raw_name in SUPERCATEGORIES:
            continue
        x, y, w, h = ann["bbox"]
        rows.append((ann["image_id"], CLASS_INDEX[RAW_TO_CANONICAL[raw_name]], x, y, w, h))
    boxes = pd.DataFrame(rows, columns=["image_id", "class_id", "x", "y", "w", "h"])
    boxes = boxes.merge(images, left_on="image_id", right_on="id").drop(columns="id")
    return images, boxes


frames = {split: to_frames(coco[split]) for split in SPLITS}

for split in SPLITS:
    _, boxes = frames[split]
    assert len(boxes) == EXPECTED_COUNTS[split]["annotations"], split

class_counts = pd.DataFrame(
    {split: frames[split][1]["class_id"].value_counts().sort_index() for split in SPLITS}
).set_axis(CLASS_NAMES)
class_counts.loc["Total"] = class_counts.sum()
class_counts